# Score the backlog with the fine-tuned FinBERT (Kaggle GPU)

Runs `pipeline/score_finbert.py`, which drains the `finbert_status='pending'`
queue in `raw_item` into `item_score`. There are ~455,000 rows waiting —
mostly the 2020-2024 GKG backfill the backtest needs scored — which is hours
on CPU and minutes here.

The queue is the state, so this is safely interruptible: every batch commits
its scores and its `finbert_status` flags in one transaction, and re-running
picks up whatever is still `pending`. Once the backlog is gone the daily
increment is ~313 rows and does not need a GPU.

## Before running

1. Settings → Accelerator → **GPU**, Internet → **On**.
2. Add Data → attach **`chloejiwon/finbert-ft-2026-08-09`** (the weights).
3. Add Data → attach the private dataset holding `pipeline/` (recipe below).
4. Add-ons → **Secrets** → add `SUPABASE_DB_URL`, and tick it for this
   notebook. This writes to the team database, so the URL cannot be inlined.

```sh
rm -rf kaggle_upload && mkdir -p kaggle_upload/pipeline
cp pipeline/__init__.py pipeline/db.py pipeline/eligibility.py \
   pipeline/labeling.py pipeline/score_finbert.py  kaggle_upload/pipeline/
```

No CSVs this time — the rows come from the database, not an upload.

In [ ]:
# --- config: the only cell you normally edit -------------------------------

CODE_DIR = "/kaggle/input/pnc-scoring"  # the dataset holding pipeline/
MODEL_DIR = "/kaggle/input/finbert-ft-2026-08-09"  # the weights

# The queue survives restarts, so a smaller LIMIT is a checkpoint, not a cap.
LIMIT = 500_000
# Rows per transaction. Higher than the module default because the Kaggle ->
# Supabase round trip, not the GPU, is what costs time here.
BATCH = 2000

WORK_DIR = "/kaggle/working/run"

In [ ]:
!pip install -q "psycopg[binary]"  # torch and transformers ship with Kaggle

In [ ]:
import os
import shutil

from kaggle_secrets import UserSecretsClient

# The DB URL is a secret, not a config value: it grants write access to the
# team's database and a notebook is shareable.
os.environ["SUPABASE_DB_URL"] = UserSecretsClient().get_secret("SUPABASE_DB_URL")

attached = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []
for path in (CODE_DIR, MODEL_DIR):
    assert os.path.isdir(path), f"{path} not attached. Attached: {attached}"

# /kaggle/input is read-only and the package has to be importable from cwd.
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_DIR, WORK_DIR)
os.chdir(WORK_DIR)

absent = [
    p
    for p in (
        "pipeline/__init__.py",
        "pipeline/db.py",
        "pipeline/eligibility.py",
        "pipeline/labeling.py",
        "pipeline/score_finbert.py",
    )
    if not os.path.exists(p)
]
assert not absent, f"missing from the upload: {absent}"
print("staged:", WORK_DIR)

In [ ]:
# Fail here, before loading a model or writing a row, if any of it is wrong.
import json

from pipeline import db
from pipeline.labeling import LABELS

with open(f"{MODEL_DIR}/metrics.json", encoding="utf-8") as f:
    metrics = json.load(f)
print("model_version:", metrics["model_version"])

conn = db.connect()
with conn.cursor() as cur:
    cur.execute("SELECT finbert_status s, count(*) n FROM raw_item GROUP BY 1")
    print("queue:", {r["s"]: r["n"] for r in cur.fetchall()})
    cur.execute("SELECT count(*) n FROM item_score")
    print("already scored:", cur.fetchone()["n"])
conn.close()
print("labels:", LABELS)

In [ ]:
!python -m pipeline.score_finbert \
    --model-dir "$MODEL_DIR" \
    --limit "$LIMIT" \
    --batch "$BATCH"

In [ ]:
# What landed. A queue that is not empty is fine — re-run this notebook.
conn = db.connect()
with conn.cursor() as cur:
    cur.execute("SELECT finbert_status s, count(*) n FROM raw_item GROUP BY 1")
    print("queue:", {r["s"]: r["n"] for r in cur.fetchall()})
    cur.execute("SELECT label, count(*) n FROM item_score GROUP BY 1")
    dist = {r["label"]: r["n"] for r in cur.fetchall()}
    total = sum(dist.values()) or 1
    print("scored:", total)
    for label in LABELS:
        print(f"  {label:9} {dist.get(label, 0):7}  {dist.get(label, 0) / total:6.1%}")
    cur.execute("""SELECT r.title, s.label, s.probs
                   FROM item_score s JOIN raw_item r ON r.id = s.raw_item_id
                   WHERE s.label <> 'neutral' LIMIT 5""")
    print("\ndirectional samples:")
    for r in cur.fetchall():
        print(f"  {r['label']:9} {r['title'][:66]}")
conn.close()

## Reading the class distribution

Expect it to look *more* neutral than the corpus truly is. The model trained
on 38 `negative` examples and distills a labeler that under-calls direction,
which is why `item_score.probs` keeps the full distribution: the aggregation
layer is meant to lower the directional threshold rather than take argmax.
A low `negative` share here is expected and is not evidence the corpus is calm.

See `scoring/DESIGN.md`, "Training on a champion that missed the criteria".